## Basic examples with PlantCaduceus

### Setup environment

In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


## Install specific pytorch
We are installing a specific version of PyTorch (2.3.1) to resolve some errors when loading mamba-ssm, The most critical part is `--index-url https://download.pytorch.org/whl/cu121`. This command tells pip to NOT use the default repository. Instead, it downloads a special version of PyTorch that was pre-compiled specifically for systems with a CUDA 12.1 driver.

#### Why is this necessary?
Google Colab has its own system-level NVIDIA driver (e.g., CUDA 12.5). By installing a PyTorch build that is aware of a compatible CUDA version (12.1 works perfectly with a 12.5 driver), we ensure that all parts of the library, including low-level components like Triton, can find the correct drivers and compile code successfully during model inference. This fixes both the `libcuda.so` and the `ModuleNotFoundError` errors.

In [ ]:
!pip3 install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121

Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 781.0/781.0 MB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 46.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 70.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 66.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 38.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 18.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 8.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 14.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196

In [ ]:
!pip3 install mamba-ssm==2.2.2 transformers==4.40.0 git+https://github.com/dridk/PyVCF3.git@master scipy==1.12.0 biopython xgboost==2.0.3 scikit-learn==1.4.0 matplotlib

  Cloning https://github.com/dridk/PyVCF3.git (to revision master) to /tmp/pip-req-build-0jnwp97t
  Running command git clone --filter=blob:none --quiet https://github.com/dridk/PyVCF3.git /tmp/pip-req-build-0jnwp97t
  Resolved https://github.com/dridk/PyVCF3.git to commit 1fb3789153d1d8e28e2cedf121399f276b5f312a
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 7.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 12.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.4/38.4 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.1/297.1 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12

🔄 You may need to restart the session after running the above setup cells.

Once restarted, you’re all set to start exploring the PlantCAD model!

## Testing if mamba-ssm is installed successfully

In [ ]:
# Test core dependencies
import torch
from mamba_ssm import Mamba
from transformers import AutoTokenizer, AutoModelForMaskedLM
import pandas as pd

device = 'cuda:0'

# Test PlantCAD model loading
tokenizer = AutoTokenizer.from_pretrained('kuleshov-group/PlantCaduceus_l32')
model = AutoModelForMaskedLM.from_pretrained('kuleshov-group/PlantCaduceus_l32', trust_remote_code=True)
model.to(device)
print("✅ Installation successful!")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/754 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/419 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

modeling_caduceus.py: 0.00B [00:00, ?B/s]

modeling_rcps.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/yairschiff/caduceus_base:
- modeling_rcps.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/yairschiff/caduceus_base:
- modeling_caduceus.py
- modeling_rcps.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/902M [00:00<?, ?B/s]

✅ Installation successful!


## Play around with the model inputs and outputs

In [ ]:
# Example plant DNA sequence (512bp max)
sequence = "CTTAATTAATATTGCCTTTGTAATAACGCGCGAAACACAAATCTTCTCTGCCTAATGCAGTAGTCATGTGTTGACTCCTTCAAAATTTCCAAGAAGTTAGTGGCTGGTGTGTCATTGTCTTCATCTTTTTTTTTTTTTTTTTAAAAATTGAATGCGACATGTACTCCTCAACGTATAAGCTCAATGCTTGTTACTGAAACATCTCTTGTCTGATTTTTTCAGGCTAAGTCTTACAGAAAGTGATTGGGCACTTCAATGGCTTTCACAAATGAAAAAGATGGATCTAAGGGATTTGTGAAGAGAGTGGCTTCATCTTTCTCCATGAGGAAGAAGAAGAATGCAACAAGTGAACCCAAGTTGCTTCCAAGATCGAAATCAACAGGTTCTGCTAACTTTGAATCCATGAGGCTACCTGCAACGAAGAAGATTTCAGATGTCACAAACAAAACAAGGATCAAACCATTAGGTGGTGTAGCACCAGCACAACCAAGAAGGGAAAAGATCGATGATCG"
device = 'cuda:0'
# Get embeddings
encoding = tokenizer.encode_plus(
            sequence,
            return_tensors="pt",
            return_attention_mask=False,
            return_token_type_ids=False
        )

input_ids = encoding["input_ids"].to(device)
with torch.inference_mode():
    outputs = model(input_ids=input_ids, output_hidden_states=True)

embeddings = outputs.hidden_states[-1]
print(f"Embedding shape: {embeddings.shape}")  # [batch_size, seq_len, embedding_dim]

Embedding shape: torch.Size([1, 512, 2048])


## Averaging forward and reverse embeddings

Given that PlantCaduceus has bi-directionality and reverse complement equivariance, so the first half of embedding is for forward sequences and the sencond half is for reverse complemented sequences, we need to average the embeddings before working on downstream classifier

In [ ]:
embeddings = embeddings.to(torch.float32).cpu().numpy()

In [ ]:
hidden_size = embeddings.shape[-1] // 2
forward = embeddings[..., 0:hidden_size]
reverse = embeddings[..., hidden_size:]
reverse = reverse[..., ::-1]
averaged_embeddings = (forward + reverse) / 2
print(averaged_embeddings.shape)

(1, 512, 1024)


In [ ]:
averaged_embeddings

array([[[-3.8904522e-04,  3.7918665e-04, -1.8403490e-04, ...,
          1.3309499e-03, -2.0754691e-03,  1.8709041e-03],
        [-3.6927901e-04,  2.7591454e-05,  4.7134439e-05, ...,
          1.5059398e-03, -1.9944913e-03,  2.0623294e-05],
        [-6.1555754e-04, -8.6638815e-05,  8.0927493e-06, ...,
          1.5799263e-03, -1.7452659e-03, -6.3965283e-04],
        ...,
        [ 4.2927015e-05, -1.6775541e-04,  8.2830720e-06, ...,
          1.6421943e-03, -3.4668043e-03,  3.2284430e-03],
        [ 4.2118662e-04,  1.1002575e-04, -6.5881955e-05, ...,
          1.7672596e-03, -3.1791721e-03,  3.7515883e-03],
        [ 4.0431562e-04,  3.5806536e-04,  3.3638283e-04, ...,
         -3.5436108e-04, -3.3066380e-03,  1.7878374e-03]]], dtype=float32)

### Masked token prediction

In [ ]:
pos = 255
sequence[pos] # the true base of this position is A

'A'

In [ ]:
input_ids[0, pos] = tokenizer.mask_token_id
with torch.inference_mode():
    outputs = model(input_ids=input_ids)

In [ ]:
nucleotides = list('acgt')
logits = outputs.logits
logits = logits[:, pos, [tokenizer.get_vocab()[nc] for nc in nucleotides]]
probs = torch.nn.functional.softmax(logits.cpu(), dim=1).numpy()

In [ ]:
probs

array([[9.99908686e-01, 1.34476395e-05, 3.83771403e-05, 3.94168528e-05]],
      dtype=float32)

In [ ]:
df = pd.DataFrame(dict(nucleotides = nucleotides, probs = probs[0]))

### The base A got the highest probability ✌🏻

In [ ]:
df

,nucleotides,probs
0,a,0.999909
1,c,0.000013
2,g,0.000038
3,t,0.000039


## Run some examples on zero-shot mutation effect prediction

In [ ]:
# clone PlantCAD repo
!git clone https://github.com/kuleshov-group/PlantCaduceus.git

Cloning into 'PlantCaduceus'...
remote: Enumerating objects: 401, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 401 (delta 30), reused 53 (delta 17), pack-reused 330 (from 1)
Receiving objects: 100% (401/401), 34.41 MiB | 12.33 MiB/s, done.
Resolving deltas: 100% (150/150), done.


In [ ]:
!wget https://download.maizegdb.org/Zm-B73-REFERENCE-NAM-5.0/Zm-B73-REFERENCE-NAM-5.0.fa.gz
!gunzip Zm-B73-REFERENCE-NAM-5.0.fa.gz

--2025-08-04 19:59:58--  https://download.maizegdb.org/Zm-B73-REFERENCE-NAM-5.0/Zm-B73-REFERENCE-NAM-5.0.fa.gz
Resolving download.maizegdb.org (download.maizegdb.org)... 104.26.11.112, 172.67.74.80, 104.26.10.112, ...
Connecting to download.maizegdb.org (download.maizegdb.org)|104.26.11.112|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 645548281 (616M) [application/x-gzip]
Saving to: ‘Zm-B73-REFERENCE-NAM-5.0.fa.gz’

Zm-B73-REFERENCE-NA 100%[===================>] 615.64M  15.6MB/s    in 46s     

2025-08-04 20:00:45 (13.2 MB/s) - ‘Zm-B73-REFERENCE-NAM-5.0.fa.gz’ saved [645548281/645548281]



In [ ]:
!python PlantCaduceus/src/zero_shot_score.py \
    -input-vcf PlantCaduceus/examples/example_maize_snp.vcf \
    -input-fasta Zm-B73-REFERENCE-NAM-5.0.fa \
    -output scored_variants.vcf \
    -model 'kuleshov-group/PlantCaduceus_l32' \
    -device 'cuda:0'

2025-08-04 20:01:16 - INFO - Reading input data from PlantCaduceus/examples/example_maize_snp.vcf
2025-08-04 20:01:32 - INFO - Loading model and tokenizer from kuleshov-group/PlantCaduceus_l32
2025-08-04 20:01:32 - INFO - Using float16 as the GPU supports sm_60 or higher.
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
2025-08-04 20:01:40 - INFO - Creating data loader
2025-08-04 20:01:40 - INFO - Creating DataLoader with batch size 128
2025-08-04 20:01:40 - INFO - Extracting lo

🎉 Nice work! The variants have been scored, and the results are now embedded in the VCF’s INFO field.

In [ ]:
!grep -v '#' scored_variants.vcf | awk  -v OFS='\t' '{print $1,$2,$4,$5,$8}' | head -n 10

chr1	2520559	A	G	MQ=59.93;TYPE=missense_variant;plantCAD_zero_shot=-1.5800781
chr1	2520575	C	T	MQ=36.74;TYPE=missense_variant;plantCAD_zero_shot=-0.35058594
chr1	2520576	G	A	MQ=59.92;TYPE=synonymous_variant;plantCAD_zero_shot=0.4399414
chr1	2520581	T	C	MQ=57.65;TYPE=missense_variant;plantCAD_zero_shot=-10.068359
chr1	2520610	T	C	MQ=56.12;TYPE=missense_variant;plantCAD_zero_shot=-5.67041
chr1	2520616	G	A	MQ=59.95;TYPE=missense_variant;plantCAD_zero_shot=-1.3505859
chr1	2520643	G	C	MQ=59.89;TYPE=intron_variant;plantCAD_zero_shot=0.16894527
chr1	2520704	C	A	MQ=59.01;TYPE=intron_variant;plantCAD_zero_shot=0.24725346
chr1	2520720	A	G	MQ=59.83;TYPE=intron_variant;plantCAD_zero_shot=0.031494107
chr1	2520825	T	C	MQ=59.19;TYPE=intron_variant;plantCAD_zero_shot=-0.29833987
